# 01. RandomForest 오프셋 안정성 검증

기존 RandomForest OOF 예측에 여러 상수 오프셋을 적용하고 2022~2024년별 Brier Score를 비교합니다. Public 리더보드가 아닌 시간 기반 OOF 결과로 보정값의 안정성을 판단합니다.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'open' / 'data').exists() and (ROOT.parent / 'open' / 'data').exists():
    ROOT = ROOT.parent
EXPERIMENT_DIR = ROOT / '0811'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OOF_CANDIDATES = [
    RESULTS_DIR / '01_rf_oof.npz',
    ROOT / '0726' / 'results' / '01_rf_oof.npz',
    ROOT / '0826' / 'results' / '01_rf_oof.npz',
    Path('/content/drive/MyDrive/baseball-results/01_rf_oof.npz'),
]
OOF_PATH = next((path for path in OOF_CANDIDATES if path.exists()), None)
if OOF_PATH is None:
    searched = '\n'.join(f'- {path}' for path in OOF_CANDIDATES)
    raise FileNotFoundError(
        '01_rf_oof.npz를 찾지 못했습니다. 0726/01_time_cv_random_forest.ipynb를 '
        f'먼저 실행하세요.\n검색 위치:\n{searched}'
    )
print('OOF:', OOF_PATH)

In [ ]:
data = np.load(OOF_PATH)
y = data['y'].astype(float)
raw_prediction = data['prediction'].astype(float)
year = data['year'].astype(int)

required_years = [2022, 2023, 2024]
missing_years = [value for value in required_years if value not in set(year)]
if missing_years:
    raise ValueError(f'OOF에 필요한 연도가 없습니다: {missing_years}')
if not (len(y) == len(raw_prediction) == len(year)):
    raise ValueError('OOF 배열 길이가 서로 다릅니다.')

def metrics(y_true, prediction):
    prediction = np.clip(np.asarray(prediction, dtype=float), 0.0, 1.0)
    y_true = np.asarray(y_true, dtype=float)
    rate = float(y_true.mean())
    brier = float(np.mean((prediction - y_true) ** 2))
    reference = rate * (1.0 - rate)
    skill = 1.0 - brier / reference if reference > 0 else 0.0
    return {
        'n': int(len(y_true)),
        'target_rate': rate,
        'prediction_mean': float(prediction.mean()),
        'brier': brier,
        'competition_score': max(0.0, 100000.0 * skill),
    }

print('rows:', f'{len(y):,}')
print('years:', dict(zip(*np.unique(year, return_counts=True))))
print('analytic overall offset:', float(np.mean(y - raw_prediction)))

## 오프셋 탐색

현재 제출값 `-0.01056616`과 사전 지정 후보를 포함하되, 결과표는 동일 구간의 촘촘한 grid도 함께 평가합니다.

In [ ]:
named_candidates = np.array([
    -0.016, -0.013, -0.01056616, -0.010, -0.008, -0.005, 0.0
])
dense_grid = np.arange(-0.030, 0.0101, 0.0005)
offsets = np.unique(np.round(np.concatenate([named_candidates, dense_grid]), 8))

rows = []
for offset in offsets:
    calibrated = np.clip(raw_prediction + offset, 0.0, 1.0)
    overall = metrics(y, calibrated)
    row = {
        'offset': float(offset),
        'overall_brier': overall['brier'],
        'overall_score': overall['competition_score'],
        'overall_prediction_mean': overall['prediction_mean'],
    }
    fold_briers = []
    fold_scores = []
    for valid_year in required_years:
        mask = year == valid_year
        fold = metrics(y[mask], calibrated[mask])
        row[f'brier_{valid_year}'] = fold['brier']
        row[f'score_{valid_year}'] = fold['competition_score']
        row[f'prediction_mean_{valid_year}'] = fold['prediction_mean']
        row[f'target_rate_{valid_year}'] = fold['target_rate']
        fold_briers.append(fold['brier'])
        fold_scores.append(fold['competition_score'])
    row['mean_fold_brier'] = float(np.mean(fold_briers))
    row['mean_fold_score'] = float(np.mean(fold_scores))
    row['worst_year_score'] = float(np.min(fold_scores))
    row['score_std'] = float(np.std(fold_scores))
    rows.append(row)

result = pd.DataFrame(rows).sort_values(
    ['mean_fold_brier', 'score_std'], ascending=[True, True]
).reset_index(drop=True)
result.to_csv(RESULTS_DIR / '01_offset_grid.csv', index=False, encoding='utf-8-sig')
result.head(15)

In [ ]:
selected = result.iloc[0]
year_optima = {}
for valid_year in required_years:
    best = result.loc[result[f'brier_{valid_year}'].idxmin()]
    year_optima[str(valid_year)] = {
        'offset': float(best['offset']),
        'brier': float(best[f'brier_{valid_year}']),
        'score': float(best[f'score_{valid_year}']),
    }

comparison_offsets = [0.0, -0.005, -0.008, -0.010, -0.01056616, -0.013, -0.016]
comparison = result[result['offset'].isin(comparison_offsets)].copy()
comparison = comparison.sort_values('offset', ascending=False)
display(comparison[[
    'offset', 'mean_fold_brier', 'overall_score',
    'score_2022', 'score_2023', 'score_2024',
    'worst_year_score', 'score_std'
]])

payload = {
    'source_oof': str(OOF_PATH),
    'selection_rule': 'minimum unweighted mean of 2022-2024 fold Brier scores',
    'public_scores_for_reference_only': {
        'raw': 549.6413938266,
        'offset_minus_0.01056616': 675.5498778599,
    },
    'analytic_offsets_by_year': {
        str(valid_year): float(np.mean(
            y[year == valid_year] - raw_prediction[year == valid_year]
        ))
        for valid_year in required_years
    },
    'grid_optima_by_year': year_optima,
    'recommended': {key: (float(value) if isinstance(value, (np.floating, float)) else int(value))
                    for key, value in selected.to_dict().items()},
}
(RESULTS_DIR / '01_offset_stability.json').write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('권장 offset:', payload['recommended']['offset'])
print('평균 Fold Brier:', payload['recommended']['mean_fold_brier'])
print('저장:', RESULTS_DIR / '01_offset_grid.csv')
print('저장:', RESULTS_DIR / '01_offset_stability.json')

## 판단 기준

- 권장값과 `-0.01056616`의 평균 Brier 차이가 매우 작으면 기존 값을 유지합니다.
- 연도별 최적 오프셋의 방향이나 크기가 크게 다르면 상수 보정이 불안정하다고 판단합니다.
- Public Score는 이미 관측된 참고값으로만 기록하며 오프셋 선택 기준으로 사용하지 않습니다.